# Notebook 2: nested hyperparameter selection, and what it cannot buy you

Companion to `REM_Turku_handoff.ipynb`; read that first.

**The short version, so you can decide whether to spend GPU here at all:** we measured,
three independent ways, that config selection extracts nothing at this data scale
(113 awakenings):

1. A 14-config sweep reproduced library defaults to 0.0006 (0.6462 vs 0.6456).
2. The dev-split spread could not rank the configs (differences inside split noise).
3. Nested selection changed nothing on any of five arms across two architectures
   (deltas -0.020 to +0.046, all inside fold-to-fold sd 0.08-0.16), against a
   pre-registered prediction that it would LOWER the numbers. It did not, because there
   was no optimism to remove: **the 12 sampled configs span 0.39-0.63 on inner
   validation while the outer sd is of the same order.** Selection is choosing among
   configs separated by less than the noise.

The same pattern is appearing on 101-Nights body_action (inner winners 0.64-0.65
delivering 0.38-0.46 on held-out folds).


## Why nesting matters even when it changes nothing

A sweep that selects on one dev split and then reports accuracy from that same split
family produces a number whose optimism you cannot bound. Nesting (select INSIDE each
training fold, score ONCE on the held-out subject) makes the estimate honest. Here the
honest estimate equals the naive one, which is itself the finding: the selection signal
is below the noise floor, measured.

Our full runs used budget=12, 3 inner folds, 17 outer folds: 8-12 h per
(architecture, target) arm on an H100. The default below is budget=3 with one target so
the cell finishes on a free Colab GPU. Do not expect the number to move; the value is
the diagnostic printed at the end.


In [ ]:
import numpy as np


def sample_configs(space, n, rng):
    """n configs from {key: [values]}. A config missing a key is a bug in the space,
    not something to paper over at scoring time."""
    out = []
    for _ in range(n):
        out.append({k: v[rng.integers(len(v))] for k, v in space.items()})
    return out


GRID = {
    "lr": [1e-4, 3e-4, 1e-3, 3e-3],
    "epochs": [20, 30, 45],
    "batch": [32, 64, 128],
}


In [ ]:
def nested_evaluate(make_fit_predict, target, budget=3, inner_folds=3, seed=0,
                    verbose=True):
    """Nested selection with the guards our first run taught us to add:

    - a config that fails scores NaN, never 0.0: twelve failures tying at 0.0 lets
      argmax silently return config 0 and the arm completes with a fake winner;
    - if every config fails, the run STOPS rather than reporting anything;
    - the diagnostic (inner span vs outer sd) is printed because it is the result."""
    from REM_Turku_handoff_lib import load, bal   # or paste the harness cell above
    rng = np.random.default_rng(seed)
    configs = sample_configs(GRID, budget, rng)
    raise NotImplementedError(
        "Wire in the harness from notebook 1 (its evaluate/load cells), then delete "
        "this line. Kept explicit so nobody runs a half-wired sweep by accident.")


## The diagnostic that costs nothing

Before spending GPU-hours, print, per outer fold: the min/max inner score across
configs (the span) and, at the end, the sd of outer scores across folds. If the span
sits inside the outer sd, selection cannot rank configs on this data and the tuned
number will equal the default number, as it did for us on five of five arms. That
one print is the honest answer to "did you tune it?".
